In [1]:
import numpy as np
from itertools import product

CB_PARAM_SPACE = {
    # model capacity
    'clf__depth':          [4, 6, 8],
    'clf__iterations':     [400, 800, 1200],
    'clf__learning_rate':  [0.03, 0.1],
    'clf__l2_leaf_reg':    [1.0, 3.0, 10.0],
    # randomness / regularization
    'clf__subsample':      [0.7, 0.9, 1.0],
    'clf__rsm':            [0.7, 0.9, 1.0],
    'clf__auto_class_weights': [None, 'Balanced'],
}

RNG_SEED = 42
def make_param_grid(space, max_combos=None, seed=RNG_SEED):
    keys = list(space.keys())
    vals = [space[k] for k in keys]
    grid = [dict(zip(keys, v)) for v in product(*vals)]

    if max_combos is not None and len(grid) > max_combos:
        rng = np.random.default_rng(seed)
        keep_idx = rng.choice(len(grid), size=max_combos, replace=False)
        grid = [grid[i] for i in keep_idx]
    return grid

CB_PARAM_GRID = make_param_grid(CB_PARAM_SPACE, max_combos=40, seed=RNG_SEED)

In [3]:
CB_PARAM_GRID[0]

{'clf__depth': 8,
 'clf__iterations': 800,
 'clf__learning_rate': 0.03,
 'clf__l2_leaf_reg': 3.0,
 'clf__subsample': 1.0,
 'clf__rsm': 0.9,
 'clf__auto_class_weights': 'Balanced'}

In [1]:
from transformers import AutoFeatureExtractor, ASTForAudioClassification
from datasets import load_dataset
import torch

dataset = load_dataset("hf-internal-testing/librispeech_asr_demo", "clean", split="validation")
dataset = dataset.sort("id")
sampling_rate = dataset.features["audio"].sampling_rate

feature_extractor = AutoFeatureExtractor.from_pretrained("MIT/ast-finetuned-audioset-10-10-0.4593")
model = ASTForAudioClassification.from_pretrained("MIT/ast-finetuned-audioset-10-10-0.4593")

# audio file is decoded on the fly
# inputs = feature_extractor(dataset[0]["audio"]["array"], sampling_rate=sampling_rate, return_tensors="pt")

# with torch.no_grad():
#     logits = model(**inputs).logits

In [2]:
dataset[0]["audio"]["array"]

: 

In [2]:
model

ASTForAudioClassification(
  (audio_spectrogram_transformer): ASTModel(
    (embeddings): ASTEmbeddings(
      (patch_embeddings): ASTPatchEmbeddings(
        (projection): Conv2d(1, 768, kernel_size=(16, 16), stride=(10, 10))
      )
      (dropout): Dropout(p=0.0, inplace=False)
    )
    (encoder): ASTEncoder(
      (layer): ModuleList(
        (0-11): 12 x ASTLayer(
          (attention): ASTAttention(
            (attention): ASTSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
            )
            (output): ASTSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.0, inplace=False)
            )
          )
          (intermediate): ASTIntermediate(
            (dense): Linear(in_features=768, out_features=3072, bias=T

In [1]:
import torchaudio.compliance.kaldi as ta_kaldi

In [14]:
config_ast = model.config

In [15]:
del config_ast.label2id
del config_ast.id2label

In [16]:
model.config

ASTConfig {
  "architectures": [
    "ASTForAudioClassification"
  ],
  "attention_probs_dropout_prob": 0.0,
  "dtype": "float32",
  "frequency_stride": 10,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.0,
  "hidden_size": 768,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "layer_norm_eps": 1e-12,
  "max_length": 1024,
  "model_type": "audio-spectrogram-transformer",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "num_mel_bins": 128,
  "patch_size": 16,
  "qkv_bias": true,
  "time_stride": 10,
  "transformers_version": "4.57.1"
}

In [ ]:
label2id

In [ ]:
id2label

In [7]:
list((inputs.keys()))

['input_values']

In [9]:
inputs['input_values'].shape

torch.Size([1, 1024, 128])